In [1]:

import os
os.environ["PYSPARK_SUBMIT_ARGS"] = (
    "--packages "
    "com.scylladb:spark-scylladb-connector_2.13:4.1.4,"
    "org.scala-lang.modules:scala-collection-compat_2.13:2.12.0 "
    "pyspark-shell"
)
import time
from pyspark.sql import SparkSession, functions as F

KEYSPACE = "birdy"
spark = (
    SparkSession.builder.appName("tp4-birds-vs-datacenters")
    .config("spark.cassandra.connection.host", "cassandra")
    .config("spark.cassandra.connection.port", "9042")
    .config("spark.sql.extensions", "com.datastax.spark.connector.CassandraSparkExtensions")
    .config("spark.sql.catalog.cass", "com.datastax.spark.connector.datasource.CassandraCatalog")
    .getOrCreate()
)
print("Spark", spark.version, "| UI:", spark.sparkContext.uiWebUrl)

Spark 4.2.0 | UI: http://4c8b377d2bc6:4040


In [2]:
spark.sql("SHOW NAMESPACES IN cass").show()
spark.sql(f"SHOW TABLES IN cass.{KEYSPACE}").show()


+---------+
|namespace|
+---------+
|    birdy|
+---------+

+---------+--------------------+-----------+
|namespace|           tableName|isTemporary|
+---------+--------------------+-----------+
|    birdy|   bird_observations|      false|
|    birdy|bird_observations...|      false|
|    birdy|         datacenters|      false|
+---------+--------------------+-----------+



In [3]:
def read_cassandra(table):
    return (spark.read.format("org.apache.spark.sql.cassandra")
            .options(keyspace=KEYSPACE, table=table).load())

birds = read_cassandra("bird_observations")
datacenters = read_cassandra("datacenters")
print(f"Keyspace: {KEYSPACE} | tables: bird_observations, datacenters")
birds.printSchema()
datacenters.printSchema()

Keyspace: birdy | tables: bird_observations, datacenters
root
 |-- id_bucket: long (nullable = false)
 |-- observation_id: long (nullable = true)
 |-- common_name: string (nullable = true)
 |-- geohash: string (nullable = true)
 |-- latitude: double (nullable = true)
 |-- longitude: double (nullable = true)
 |-- observed_on: date (nullable = true)
 |-- scientific_name: string (nullable = true)

root
 |-- external_id: string (nullable = false)
 |-- latitude: double (nullable = true)
 |-- longitude: double (nullable = true)
 |-- name: string (nullable = true)
 |-- opened_on: date (nullable = true)
 |-- operator: string (nullable = true)



In [4]:

birds.show(5, truncate=False)
datacenters.show(5, truncate=False)
print("bird observations:", birds.count())
print("datacenters:", datacenters.count())

+---------+--------------+------------------+-------+-------------+--------------+-----------+-------------------+
|id_bucket|observation_id|common_name       |geohash|latitude     |longitude     |observed_on|scientific_name    |
+---------+--------------+------------------+-------+-------------+--------------+-----------+-------------------+
|402      |402595178     |Canada Goose      |f0pr   |46.3424361007|-79.4909356778|2026-09-22 |Branta canadensis  |
|402      |402595202     |White Ibis        |djj2   |28.2072428572|-82.2891363326|2026-09-19 |Eudocimus albus    |
|402      |402595203     |Striated Pardalote|r235   |-42.98858538 |147.71750043  |2026-09-23 |Pardalotus striatus|
|402      |402595206     |Swainson's Thrush |dphr   |40.76782682  |-83.8320484   |2026-09-22 |Catharus ustulatus |
|402      |402595207     |Mallard           |dnup   |39.3201116667|-84.2721866667|2026-09-22 |Anas platyrhynchos |
+---------+--------------+------------------+-------+-------------+-------------

**Données :** `birdy.bird_observations` (observations d'oiseaux iNaturalist, partitionnées par `id_bucket = observation_id // 1 000 000`) et `birdy.datacenters` (sites OpenStreetMap).

**Colonnes clés :** `common_name`, `scientific_name`, `observed_on`, `latitude`, `longitude` ; pour les datacenters, `latitude` et `longitude`.

**Question métier :** comment le nombre d'observations d'oiseaux évolue-t-il, mois par mois et au fil des ans, dans les zones où se trouve un datacenter ?

In [5]:
birds_sel = birds.select("observation_id", "common_name", "scientific_name",
                         "observed_on", "latitude", "longitude")
dc_sel = datacenters.select("external_id", "name", "operator", "latitude", "longitude")

In [6]:
EUROPE = (F.col("latitude").between(35, 71)) & (F.col("longitude").between(-11, 40))
birds_located = birds_sel.where(F.col("latitude").isNotNull() & F.col("longitude").isNotNull())
birds_recent = birds_located.where(F.col("observed_on") >= F.lit("2026-01-01"))
birds_europe = birds_recent.where(EUROPE)
print("located:", birds_located.count(), "| since 2026:", birds_recent.count(),
      "| since 2026 in Europe:", birds_europe.count())

located: 459122 | since 2026: 417611 | since 2026 in Europe: 84440


In [7]:
def with_grid(df):
    # 1° x 1° cell (~111 km x 111 km at the equator), shared by birds and datacenters
    return df.withColumn(
        "grid_cell",
        F.concat_ws("_", F.floor("latitude").cast("int"), F.floor("longitude").cast("int")),
    )

birds_enriched = (
    with_grid(birds_located)
    .withColumn("days_since_observation", F.datediff(F.current_date(), "observed_on"))
    .withColumn("observed_month", F.date_format("observed_on", "yyyy-MM"))
)
birds_enriched.select("common_name", "observed_on", "days_since_observation",
                      "latitude", "longitude", "grid_cell").show(5, truncate=False)

+---------------------+-----------+----------------------+-------------+--------------+---------+
|common_name          |observed_on|days_since_observation|latitude     |longitude     |grid_cell|
+---------------------+-----------+----------------------+-------------+--------------+---------+
|Ring-billed Gull     |2026-09-27 |11                    |45.43385     |-73.80967     |45_-74   |
|Eurasian Tree Sparrow|2026-09-17 |21                    |54.9504309095|31.9969105348 |54_31    |
|Killdeer             |2026-09-27 |11                    |34.1562      |-84.7672366667|34_-85   |
|Canada Goose         |2026-09-23 |15                    |44.5407633333|-110.42398    |44_-111  |
|Pomarine Jaeger      |2026-09-27 |11                    |42.6887218858|-70.6219801027|42_-71   |
+---------------------+-----------+----------------------+-------------+--------------+---------+
only showing top 5 rows


In [8]:

top_species = (birds_enriched.groupBy("common_name")
               .agg(F.count("*").alias("observations"),
                    F.countDistinct("grid_cell").alias("cells"),
                    F.round(F.avg("days_since_observation"), 1).alias("avg_age_days"))
               .orderBy(F.desc("observations")))
top_species.show(10, truncate=False)

by_month = birds_enriched.groupBy("observed_month").count().orderBy(F.desc("observed_month"))
by_month.show(12)

+----------------+------------+-----+------------+
|common_name     |observations|cells|avg_age_days|
+----------------+------------+-----+------------+
|Mallard         |7049        |1055 |123.9       |
|Feral Pigeon    |4378        |986  |99.3        |
|House Sparrow   |4369        |1036 |88.8        |
|Great Egret     |4364        |1026 |136.9       |
|Great Blue Heron|4259        |582  |96.0        |
|Birds           |4217        |1308 |76.4        |
|Canada Goose    |3185        |544  |148.3       |
|Grey Heron      |3030        |686  |192.8       |
|Red-tailed Hawk |2677        |485  |106.8       |
|Turkey Vulture  |2550        |594  |91.5        |
+----------------+------------+-----+------------+
only showing top 10 rows
+--------------+------+
|observed_month| count|
+--------------+------+
|       2026-10|113230|
|       2026-09|254025|
|       2026-08| 20049|
|       2026-07|  8878|
|       2026-06|  5578|
|       2026-05|  5308|
|       2026-04|  3878|
|       2026-03|  290

In [9]:
print("birds partitions:", birds.rdd.getNumPartitions())
(birds.withColumn("partition", F.spark_partition_id())
      .groupBy("partition").count().orderBy("partition").show(50))

birds_8 = birds.repartition(8)
print("after repartition(8):", birds_8.rdd.getNumPartitions())
(birds_8.withColumn("partition", F.spark_partition_id())
        .groupBy("partition").count().orderBy("partition").show())
birds_2 = birds_8.coalesce(2)
print("after coalesce(2):", birds_2.rdd.getNumPartitions())
print("spark.sql.shuffle.partitions =", spark.conf.get("spark.sql.shuffle.partitions"))

birds partitions: 21
+---------+------+
|partition| count|
+---------+------+
|        0|112324|
|        6|113493|
|       11|163428|
|       16| 70280|
+---------+------+

after repartition(8): 8
+---------+-----+
|partition|count|
+---------+-----+
|        0|57466|
|        1|57466|
|        2|57465|
|        3|57465|
|        4|57465|
|        5|57465|
|        6|57467|
|        7|57466|
+---------+-----+

after coalesce(2): 2
spark.sql.shuffle.partitions = 200


On obtient 21 partitions Spark, mais seules quelques-unes contiennent des données (4 ici). Le connecteur découpe la lecture par plages de tokens Cassandra, et la clé de partition `id_bucket` ne prend qu'une poignée de valeurs (une par tranche d'un million d'identifiants). Le modèle Cassandra détermine donc directement la répartition, et le déséquilibre, du travail côté Spark. `repartition(8)` rééquilibre les données (8 partitions de taille égale) au prix d'un shuffle ; `coalesce(2)` réduit le nombre de partitions sans shuffle.

In [10]:
t0 = time.perf_counter()
step1 = birds.select("common_name", "observed_on", "latitude", "longitude")
step2 = step1.where(F.col("latitude").isNotNull())
step3 = step2.withColumn("observed_month", F.date_format("observed_on", "yyyy-MM"))
step4 = step3.groupBy("observed_month").count()
print(f"4 transformations defined in {time.perf_counter() - t0:.3f} s - nothing executed yet")

4 transformations defined in 0.068 s - nothing executed yet


In [11]:
t0 = time.perf_counter()
result = step4.orderBy(F.desc("count")).collect()
print(f"action collect(): {time.perf_counter() - t0:.2f} s, {len(result)} rows")
print(result[:5])

action collect(): 1.60 s, 375 rows
[Row(observed_month='2026-09', count=254321), Row(observed_month='2026-10', count=113230), Row(observed_month='2026-08', count=20114), Row(observed_month='2026-07', count=8886), Row(observed_month='2026-06', count=5580)]


 Les transformations (select, where, withColumn, groupBy) ne font que construire un plan logique, ce qui prend environ 0,07 s. Seule une action (collect, count, show) lance l'exécution (environ 1 s). Différer le calcul permet à Spark (optimiseur Catalyst) de voir toute la chaîne avant d'exécuter : il pousse les filtres et la projection jusqu'à la source (ici, le scan Cassandra ne lit que latitude et observed_on), fusionne les étapes sans shuffle dans un même stage, évite de matérialiser des résultats intermédiaires inutiles et peut recalculer une partition perdue à partir de son lignage.

In [12]:
step4.explain(mode="formatted")

== Physical Plan ==
AdaptiveSparkPlan (7)
+- HashAggregate (6)
   +- Exchange (5)
      +- HashAggregate (4)
         +- Project (3)
            +- Filter (2)
               +- BatchScan bird_observations (1)


(1) BatchScan bird_observations
Output [2]: [latitude#32, observed_on#34]
Cassandra Scan: birdy.bird_observations
 - Cassandra Filters: []
 - Requested Columns: [latitude,observed_on]

(2) Filter
Input [2]: [latitude#32, observed_on#34]
Condition : isnotnull(latitude#32)

(3) Project
Output [1]: [date_format(cast(observed_on#34 as timestamp), yyyy-MM, Some(Etc/UTC)) AS observed_month#338]
Input [2]: [latitude#32, observed_on#34]

(4) HashAggregate
Input [1]: [observed_month#338]
Keys [1]: [observed_month#338]
Functions [1]: [partial_count(1)]
Aggregate Attributes [1]: [count#349L]
Results [2]: [observed_month#338, count#350L]

(5) Exchange
Input [2]: [observed_month#338, count#350L]
Arguments: hashpartitioning(observed_month#338, 200), ENSURE_REQUIREMENTS, [plan_id=752]

(6) Has

## Analyse métier : observations par mois au fil des ans, en zone datacenter

Une **zone datacenter** est une cellule de 1° × 1° (`grid_cell`) qui contient au moins un datacenter. On ne compare pas ces zones aux autres : elles sont plus urbaines, donc plus observées (biais d'observation). On suit plutôt l'évolution dans le temps à l'intérieur de ces zones.

In [13]:
spark.sparkContext.setJobDescription("TP4 - observations par mois en zone datacenter")

# Grid cells that contain at least one datacenter (small table: ~300 rows)
dc_cells = (with_grid(dc_sel).groupBy("grid_cell")
            .agg(F.count("*").alias("datacenters")))

# Dated observations since 2020, flagged when they fall in a datacenter zone
birds_dated = (birds_enriched
               .where(F.col("observed_on") >= F.lit("2020-01-01"))
               .withColumn("year", F.year("observed_on"))
               .withColumn("month", F.month("observed_on")))

birds_flagged = (birds_dated.join(F.broadcast(dc_cells), "grid_cell", "left")
                 .withColumn("in_dc_zone", F.col("datacenters").isNotNull()))

monthly = (birds_flagged.groupBy("year", "month", "observed_month")
           .agg(F.count("*").alias("all_observations"),
                F.sum(F.col("in_dc_zone").cast("int")).alias("dc_zone_observations"))
           .withColumn("dc_zone_share_pct",
                       F.round(100 * F.col("dc_zone_observations") / F.col("all_observations"), 1))
           .orderBy("observed_month"))

monthly.explain()
monthly.show(100)

== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- Sort [observed_month#165 ASC NULLS FIRST], true, 0
   +- Exchange rangepartitioning(observed_month#165 ASC NULLS FIRST, 200), ENSURE_REQUIREMENTS, [plan_id=810]
      +- Project [year#363, month#364, observed_month#165, all_observations#366L, dc_zone_observations#367L, round((cast((dc_zone_observations#367L * 100) as double) / cast(all_observations#366L as double)), 1) AS dc_zone_share_pct#383]
         +- HashAggregate(keys=[year#363, month#364, observed_month#165], functions=[count(1), sum(cast(in_dc_zone#365 as int))])
            +- Exchange hashpartitioning(year#363, month#364, observed_month#165, 200), ENSURE_REQUIREMENTS, [plan_id=806]
               +- HashAggregate(keys=[year#363, month#364, observed_month#165], functions=[partial_count(1), partial_sum(cast(in_dc_zone#365 as int))])
                  +- Project [observed_month#165, year#363, month#364, isnotnull(datacenters#355L) AS in_dc_zone#365]
                    

In [14]:
# Year x month table: observations in datacenter zones
(birds_flagged.where("in_dc_zone")
              .groupBy("year")
              .pivot("month", list(range(1, 13)))
              .count()
              .na.fill(0)
              .orderBy("year")
              .show())

+----+---+---+---+----+----+----+----+----+-----+-----+---+---+
|year|  1|  2|  3|   4|   5|   6|   7|   8|    9|   10| 11| 12|
+----+---+---+---+----+----+----+----+----+-----+-----+---+---+
|2020| 48| 38| 20|  13|  57|  36|   9|  11|   15|    9| 10| 16|
|2021|  6| 16|  5|  39|  22|  33|  31|  10|   19|   12| 12|  9|
|2022| 17| 27| 51|  55|  53|  39|  42|  22|   27|   20|  9| 15|
|2023| 27| 50| 51|  60| 127|  72| 112|  46|   44|   93| 70| 34|
|2024| 32| 26| 69| 134| 122|  67|  53|  34|  104|   68| 58| 67|
|2025| 56| 72| 99| 341| 199| 155| 189| 167|  174|  424|306|404|
|2026|364|348|883|1275|1385|1178|1921|2763|69451|34422|  0|  0|
+----+---+---+---+----+----+----+----+----+-----+-----+---+---+



In [15]:
# Seasonality on complete years only (2020-2025): average per calendar month
seasonality = (monthly.where(F.col("year") < 2026)
               .groupBy("month")
               .agg(F.round(F.avg("dc_zone_observations"), 1).alias("avg_dc_zone_observations"),
                    F.round(F.avg("dc_zone_share_pct"), 1).alias("avg_dc_zone_share_pct"))
               .orderBy("month"))
seasonality.show()
spark.sparkContext.setJobDescription(None)

+-----+------------------------+---------------------+
|month|avg_dc_zone_observations|avg_dc_zone_share_pct|
+-----+------------------------+---------------------+
|    1|                    31.0|                 11.9|
|    2|                    38.2|                 16.8|
|    3|                    49.2|                 20.4|
|    4|                   107.0|                 22.8|
|    5|                    96.7|                 26.8|
|    6|                    67.0|                 25.3|
|    7|                    72.7|                 18.5|
|    8|                    48.3|                 11.4|
|    9|                    63.8|                 18.5|
|   10|                   104.3|                 19.3|
|   11|                    77.5|                 12.2|
|   12|                    90.8|                 13.4|
+-----+------------------------+---------------------+



**Observations :**

- **Saisonnalité dans les zones datacenter :** chaque année complète montre le même cycle. Un pic au printemps (avril-mai : migration prénuptiale et nidification), un second pic à l'automne (octobre : migration postnuptiale), et un creux en hiver et en août. En moyenne sur 2020-2025 : environ 104 observations en avril et 102 en octobre, contre 31 en janvier.
- **Part des zones datacenter (`dc_zone_share_pct`) :** elle suit le même rythme. Environ 23 à 27 % des observations mondiales d'avril à juin, contre 11 à 13 % en août, novembre et janvier. Les zones datacenter se trouvent toutes dans l'hémisphère nord tempéré (US, Europe de l'Ouest), où la migration printanière concentre les observations, alors que le total mondial inclut l'hémisphère sud et les tropiques. Cette part est bruitée avant 2025 (petits effectifs : 3,6 % à 50,9 % selon les mois).
- **Évolution au fil des ans :** les volumes augmentent fortement d'année en année (moins de 300 observations en zone datacenter en 2020 et 2021, plus de 2 500 en 2025). Ce n'est pas un signal écologique : l'ingestion récupère les observations iNaturalist des plus récentes aux plus anciennes et n'a pas fini de remonter dans le temps, et iNaturalist lui-même compte de plus en plus d'utilisateurs. Septembre-octobre 2026 (plus de 100 000 observations) reflète surtout la période de collecte.
- **Exécution :** le petit DataFrame `dc_cells` est diffusé à tous les exécuteurs (`BroadcastHashJoin`), donc pas de shuffle pour la jointure. Les `Exchange` restants viennent des `groupBy` (cellules de datacenters, agrégation mensuelle) et du tri (`Exchange rangepartitioning` pour `orderBy`).